# SWIM Sanity Check: Multi-Layer Network

Debugging notebook for the SWIM pipeline with 2D nonlinear data: y = x1^2 + sin(x2)

In [1]:
import numpy as np
import sys
sys.path.insert(0, '/Users/gizemnurdal/Workspace/swim-meets-kans/swimnetworks-paper')

from swimnetworks.dense import Dense
from swimnetworks.linear import Linear

## Step 1: Create Data

Generate training and test data for y = x1^2 + sin(x2)

In [ ]:
# Set random seed
np.random.seed(42)

# Training data: 1000 samples, 2 dimensions
N_train = 10000
D = 2
X_train = np.random.uniform(0, 2*np.pi, size=(N_train, D))
y_train = (X_train[:, 0]**2 + np.sin(X_train[:, 1])).reshape(-1, 1)

# Test data: 200 samples, same distribution
N_test = 200
X_test = np.random.uniform(0, 2*np.pi, size=(N_test, D))
y_test = (X_test[:, 0]**2 + np.sin(X_test[:, 1])).reshape(-1, 1)

print(f"Training data:")
print(f"  X_train: shape={X_train.shape}")
print(f"  y_train: shape={y_train.shape}")
print(f"\nTest data:")
print(f"  X_test: shape={X_test.shape}")
print(f"  y_test: shape={y_test.shape}")
print(f"\nData ranges:")
print(f"  X_train min/max: [{X_train.min():.4f}, {X_train.max():.4f}]")
print(f"  y_train min/max: [{y_train.min():.4f}, {y_train.max():.4f}]")

Training data:
  X_train: shape=(1000, 2)
  y_train: shape=(1000, 1)

Test data:
  X_test: shape=(200, 2)
  y_test: shape=(200, 1)

Data ranges:
  X_train min/max: [0.0202, 6.2814]
  y_train min/max: [-0.9433, 40.1219]


## Step 2: First Dense Layer

In [3]:
# First Dense layer
dense1 = Dense(
    layer_width=128,
    activation="tanh",
    parameter_sampler="tanh",
    random_seed=1
)

print("Fitting Dense layer 1...")
dense1.fit(X_train, y_train)
print("Done!")

# Transform with first Dense layer
H1 = dense1.transform(X_train)
print(f"\nH1 (hidden layer 1 output): shape={H1.shape}")
print(f"Dense1 weights: shape={dense1.weights.shape}")
print(f"Dense1 biases: shape={dense1.biases.shape}")

Fitting Dense layer 1...
Done!

H1 (hidden layer 1 output): shape=(1000, 128)
Dense1 weights: shape=(2, 128)
Dense1 biases: shape=(1, 128)


## Step 3: Second Dense Layer

In [4]:
# Second Dense layer
dense2 = Dense(
    layer_width=128,
    activation="tanh",
    parameter_sampler="tanh",
    random_seed=2
)

print("Fitting Dense layer 2...")
dense2.fit(H1, y_train)
print("Done!")

# Transform with second Dense layer
H2 = dense2.transform(H1)
print(f"\nH2 (hidden layer 2 output): shape={H2.shape}")
print(f"Dense2 weights: shape={dense2.weights.shape}")
print(f"Dense2 biases: shape={dense2.biases.shape}")

Fitting Dense layer 2...
Done!

H2 (hidden layer 2 output): shape=(1000, 128)
Dense2 weights: shape=(128, 128)
Dense2 biases: shape=(1, 128)


## Step 4: Linear Output Layer

In [5]:
# Linear output layer
linear = Linear(regularization_scale=1e-10)

print("Fitting Linear layer...")
linear.fit(H2, y_train)
print("Done!")

print(f"\nLinear weights: shape={linear.weights.shape}")
print(f"Linear biases: shape={linear.biases.shape}")

Fitting Linear layer...
Done!

Linear weights: shape=(128, 1)
Linear biases: shape=(1, 1)


## Step 5: Training Performance

In [6]:
# Predictions on training data
H1_train = dense1.transform(X_train)
H2_train = dense2.transform(H1_train)
y_pred_train = linear.transform(H2_train)

# Compute training metrics
mse_train = np.mean((y_pred_train - y_train) ** 2)
rmse_train = np.sqrt(mse_train)
mae_train = np.mean(np.abs(y_pred_train - y_train))

print(f"Training Performance:")
print(f"  MSE:  {mse_train:.6f}")
print(f"  RMSE: {rmse_train:.6f}")
print(f"  MAE:  {mae_train:.6f}")
print(f"\nPredictions sample (first 5):")
print(f"{'Actual':<12} {'Predicted':<12} {'Error':<12}")
print("-" * 36)
for i in range(5):
    error = y_pred_train[i, 0] - y_train[i, 0]
    print(f"{y_train[i, 0]:<12.6f} {y_pred_train[i, 0]:<12.6f} {error:<12.6f}")

Training Performance:
  MSE:  0.000113
  RMSE: 0.010633
  MAE:  0.007901

Predictions sample (first 5):
Actual       Predicted    Error       
------------------------------------
5.233299     5.227127     -0.006172   
20.572188    20.572773    0.000584    
1.791553     1.791486     -0.000067   
-0.612015    -0.601190    0.010825    
13.299602    13.292924    -0.006678   


## Step 6: Test Performance (Generalization Check)

In [7]:
# Predictions on test data
H1_test = dense1.transform(X_test)
H2_test = dense2.transform(H1_test)
y_pred_test = linear.transform(H2_test)

# Compute test metrics
mse_test = np.mean((y_pred_test - y_test) ** 2)
rmse_test = np.sqrt(mse_test)
mae_test = np.mean(np.abs(y_pred_test - y_test))

print(f"Test Performance:")
print(f"  MSE:  {mse_test:.6f}")
print(f"  RMSE: {rmse_test:.6f}")
print(f"  MAE:  {mae_test:.6f}")
print(f"\nGeneralization:")
print(f"  Train MSE: {mse_train:.6f}")
print(f"  Test MSE:  {mse_test:.6f}")
print(f"  Ratio (test/train): {mse_test/mse_train:.4f}")

Test Performance:
  MSE:  0.000140
  RMSE: 0.011830
  MAE:  0.009026

Generalization:
  Train MSE: 0.000113
  Test MSE:  0.000140
  Ratio (test/train): 1.2379


## Step 7: Pipeline Summary

In [8]:
print("Pipeline Architecture:")
print(f"  Input:              {D} dimensions")
print(f"  Dense1:             128 units, tanh activation")
print(f"  Dense2:             128 units, tanh activation")
print(f"  Linear:             1 unit (regression output)")
print(f"\nTotal Parameters:")
total_params = dense1.n_parameters + dense2.n_parameters + linear.n_parameters
print(f"  Dense1: {dense1.n_parameters}")
print(f"  Dense2: {dense2.n_parameters}")
print(f"  Linear: {linear.n_parameters}")
print(f"  Total:  {total_params}")

Pipeline Architecture:
  Input:              2 dimensions
  Dense1:             128 units, tanh activation
  Dense2:             128 units, tanh activation
  Linear:             1 unit (regression output)

Total Parameters:
  Dense1: 384
  Dense2: 16512
  Linear: 129
  Total:  17025
